# Chapter 03 데모 — 보안 및 자격 증명: 인증된 고객만, 안전한 자격 증명으로

> 📑 대응 강의: **03-SecurityAndIdentity.pptx** (보안 및 자격 증명 관리)

| 챕터 | 노트북 | 고객지원 에이전트에 추가되는 기능 |
|---|---|---|
| 02 | `02_Runtime_Demo` | Strands 에이전트 → **AgentCore Runtime** 서버리스 배포, 세션 격리, 버전/엔드포인트 |
| 👉 **03** | `03_SecurityIdentity_Demo` | **Cognito JWT 인바운드 인증**, 사용자 식별·감사 로그, **API 키 아웃바운드 인증**(배송 파트너 API) |
| 04 | `04_ToolsGateway_Demo` | **Code Interpreter**, MCP, **AgentCore Gateway**(Lambda→MCP 도구), **AgentCore Policy**(Cedar) |
| 05 | `05_Memory_Demo` | **AgentCore Memory** 단기/장기 메모리 → 세션을 넘어 고객을 기억 |
| 06 | `06_Observability_Evaluation_Demo` | **Observability**(트레이스/스팬/지표), **Evaluations**(온디맨드/온라인), PROD 승격 |
| 99 | `99_Cleanup` | 데모 리소스 일괄 삭제 |

### 02장에서 남은 문제
- IAM 자격 증명만 있으면 누구나 에이전트를 호출할 수 있고, 에이전트는 **누가** 요청했는지 모릅니다.
- 배송 파트너 API 처럼 **외부 서비스의 자격 증명**을 코드에 넣지 않고 쓸 방법이 필요합니다.

### 이번 챕터에서 할 일
| 구분 | 내용 | 슬라이드 |
|---|---|---|
| **인바운드 인증** | Amazon Cognito 사용자 풀 → Runtime 에 **JWT Bearer 토큰 인증** 적용 | 5–7, 21 |
| 사용자 컨텍스트 | JWT 클레임으로 고객 식별, 세션 소유자 검증, **감사 로그** | 10 |
| **아웃바운드 인증** | AgentCore Identity **API 키 자격 증명 공급자** → `@requires_api_key` | 8, 18–20 |
| 워크로드 자격 증명 | Runtime 이 자동 생성한 에이전트 ID 확인 | 16–17 |
| 실행 역할 | 런타임 IAM 역할 권한 구성 살펴보기 | 22 |

```
 고객(alice/bob) ─로그인─▶ Cognito ─JWT─┐
                                        ▼
                     AgentCore Runtime (JWT 검증 = 인바운드 인증)
                        └─ 고객지원 에이전트 (username → 사용자 컨텍스트, 감사 로그)
                              └─ track_shipment ─(API 키, AgentCore Identity = 아웃바운드 인증)─▶ 배송 파트너 API
```

## 0. 환경 준비 & 02장 결과물 불러오기

In [ ]:
import json, time
from pathlib import Path

import boto3
from demo_helpers import config
from demo_helpers.aws_setup import get_region, get_account_id, wait_for, decode_jwt_claims
from demo_helpers.invoke import invoke_with_iam, invoke_with_jwt, new_session_id, show

REGION = get_region()
ACCOUNT_ID = get_account_id()
state = config.require_state("agent_arn", "agent_id", "runtime_role_arn")
agent_arn, agent_id = state["agent_arn"], state["agent_id"]
control = boto3.client("bedrock-agentcore-control", region_name=REGION)
print("Agent ARN:", agent_arn)

## 1. 인바운드 인증: 자격 증명 공급자(IdP) 준비 (슬라이드 5–7)

| 방식 | 사용 사례 |
|---|---|
| **IAM SigV4** | AWS 내부 서비스·백엔드 간 호출 (02장에서 사용) |
| **JWT Bearer 토큰** | 최종 사용자가 앱에 로그인한 뒤 에이전트 호출 — Cognito, Okta, Entra ID 등 OIDC IdP |

데모 고객 **alice**, **bob** 이 있는 Amazon Cognito 사용자 풀을 만듭니다.

In [ ]:
from demo_helpers.aws_setup import ensure_cognito_user_pool, get_bearer_token

state = ensure_cognito_user_pool()
print("Discovery URL:", state["cognito_discovery_url"])
print("Client ID    :", state["cognito_client_id"])

In [ ]:
alice_token = get_bearer_token("alice")
bob_token = get_bearer_token("bob")

claims = decode_jwt_claims(alice_token)
print(json.dumps({k: claims[k] for k in ("iss", "client_id", "username", "sub", "token_use", "exp")}, indent=2))

JWT 의 `client_id` 는 Runtime 의 **allowedClients** 검증에, `username`/`sub` 는 에이전트의 **사용자 컨텍스트**에 쓰입니다.

## 2. 아웃바운드 인증: API 키 자격 증명 공급자 (슬라이드 8, 18–20)

에이전트가 사용자를 대신해 외부 리소스에 접근하는 3가지 패턴:
1. **AWS 서비스** → IAM 역할 (실행 역할)
2. **머신 간(M2M)** → OAuth 2.0 클라이언트 자격 증명(2LO) 또는 **API 키**
3. **사용자 위임** → OAuth 2.0 인증 코드 흐름(3LO, 사용자 동의)

배송 파트너 API 키를 **AgentCore Identity 토큰 볼트**에 저장합니다. 키는 AWS Secrets Manager 에 암호화 보관되며,
에이전트 코드·환경 변수·이미지 어디에도 키가 남지 않습니다.

In [ ]:
import secrets
from botocore.exceptions import ClientError
from bedrock_agentcore.services.identity import IdentityClient

partner_api_key = "ship-partner-" + secrets.token_hex(12)   # 데모용으로 생성한 '외부 파트너 API 키'

identity_client = IdentityClient(REGION)
try:
    provider = identity_client.create_api_key_credential_provider({
        "name": config.PARTNER_API_PROVIDER,
        "apiKey": partner_api_key,
    })
    print("✅ API 키 자격 증명 공급자 생성")
except ClientError as e:
    if e.response["Error"]["Code"] != "ConflictException":
        raise
    provider = control.get_api_key_credential_provider(name=config.PARTNER_API_PROVIDER)
    print("ℹ️  기존 공급자 재사용 (키 끝 4자리는 기존 값)")
    partner_api_key = None

print("Provider ARN:", provider["credentialProviderArn"])
if partner_api_key:
    config.save_state(partner_key_last4=partner_api_key[-4:])
print("에이전트 응답에서 확인할 키 끝 4자리:", config.load_state().get("partner_key_last4"))

> 참고 — OAuth2 공급자(슬라이드 19)도 같은 방식입니다:
> ```python
> identity_client.create_oauth2_credential_provider({
>     "name": "github-provider", "credentialProviderVendor": "GithubOauth2",
>     "oauth2ProviderConfigInput": {"githubOauth2ProviderConfig": {"clientId": "...", "clientSecret": "..."}}})
> ```
> 에이전트 코드에서는 `@requires_access_token(provider_name=..., scopes=[...], auth_flow="M2M" | "USER_FEDERATION")` 를 사용합니다.

## 3. 에이전트 코드 업데이트 (v2)

02장 코드에 `# [CH03]` 표시 부분이 추가됩니다.
- **사용자 컨텍스트**: Runtime 이 서명을 검증한 JWT 에서 `username` 을 꺼내 에이전트에 전달
- **세션 소유자 검증**: 한 세션(microVM)은 처음 사용한 고객 전용 — 다른 사용자의 세션 ID 재사용 차단
- **감사 로그**: 모든 요청을 사용자·세션·사용 도구와 함께 구조화 로그로 기록 (슬라이드 10)
- **`track_shipment`**: `@requires_api_key` 가 실행 시점에 Identity 에서 API 키를 받아옴

In [ ]:
%%writefile agent/runtime_ch03.py
"""[Chapter 03] + JWT 인바운드 인증(사용자 컨텍스트·감사 로그) + API 키 아웃바운드 인증."""
import base64
import hashlib
import json
import os
import threading
import time
import uuid

from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from bedrock_agentcore.identity.auth import requires_api_key  # [CH03]

from demo_helpers.config import MODEL_ID, PARTNER_API_PROVIDER
from demo_helpers.tools import SYSTEM_PROMPT, get_product_info, get_return_policy

REGION = os.environ.get("AWS_REGION", "us-east-1")
MICROVM_BOOT_ID = uuid.uuid4().hex[:8]

app = BedrockAgentCoreApp()


@tool
def generate_sales_report(duration_seconds: int = 20) -> str:
    """장시간 걸리는 '판매 리포트 생성' 작업을 백그라운드로 시작하고 즉시 응답합니다.

    Args:
        duration_seconds: 작업 소요 시간(초)
    """
    task_id = app.add_async_task("sales_report", {"duration": duration_seconds})

    def background_work():
        time.sleep(duration_seconds)
        app.complete_async_task(task_id)

    threading.Thread(target=background_work, daemon=True).start()
    return f"리포트 작업(ID: {task_id})을 시작했습니다. 약 {duration_seconds}초 후 완료됩니다."


# ── [CH03] 아웃바운드 인증: AgentCore Identity 에서 API 키를 받아옴 ─────────
@requires_api_key(provider_name=PARTNER_API_PROVIDER)
def _get_partner_api_key(*, api_key: str = "") -> str:
    return api_key  # 워크로드 액세스 토큰 → GetResourceApiKey 로 받은 키가 주입됨


@tool
def track_shipment(order_id: str) -> str:
    """주문 번호로 배송 상태를 조회합니다 (외부 배송 파트너 API 호출).

    Args:
        order_id: 주문 번호 (예: ORD-1001)
    """
    api_key = _get_partner_api_key()
    # 실제 환경이라면: requests.get(f"https://api.partner.example/track/{order_id}", headers={"x-api-key": api_key})
    statuses = ["상품 준비 중", "배송 중 (서울 허브 도착)", "배송 완료"]
    status = statuses[int(hashlib.md5(order_id.encode()).hexdigest(), 16) % len(statuses)]
    return f"주문 {order_id}: {status} (파트너 API 키 ****{api_key[-4:]} 로 인증됨)"


# ── [CH03] 인바운드 인증: Runtime 이 검증한 JWT 에서 사용자 컨텍스트 추출 ────
def get_caller_claims(context) -> dict:
    auth = (context.request_headers or {}).get("Authorization", "")
    if not auth.startswith("Bearer "):
        return {}
    payload = auth.split(".")[1]
    payload += "=" * (-len(payload) % 4)
    return json.loads(base64.urlsafe_b64decode(payload))  # 서명 검증은 Runtime 이 이미 수행


_agent = None
_session_owner = None  # [CH03] 이 세션(microVM)의 소유자


@app.entrypoint
def invoke(payload, context):
    global _agent, _session_owner
    claims = get_caller_claims(context)
    username = claims.get("username", "anonymous")

    if _session_owner and _session_owner != username:  # [CH03] 세션 소유자 검증
        print(json.dumps({"type": "AUDIT", "event": "SESSION_OWNER_MISMATCH", "user": username,
                          "session_id": context.session_id}))
        return {"response": "⛔ 이 세션은 다른 사용자에게 속해 있어 사용할 수 없습니다.", "user": username}

    if _agent is None:
        _session_owner = username
        _agent = Agent(
            model=BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=0.3),
            tools=[get_product_info, get_return_policy, generate_sales_report, track_shipment],
            system_prompt=SYSTEM_PROMPT + f"\n현재 로그인한 고객 ID: {username}\n",
            callback_handler=None,
        )

    prompt = payload.get("prompt", "")
    result = _agent(prompt)
    tools_used = list(result.metrics.tool_metrics.keys())

    print(json.dumps({  # [CH03] 감사 로그 → CloudWatch Logs
        "type": "AUDIT", "user": username, "sub": claims.get("sub"),
        "session_id": context.session_id, "prompt": prompt[:200], "tools_used_in_session": tools_used,
    }, ensure_ascii=False))

    return {
        "response": str(result),
        "user": username,
        "session_id": context.session_id,
        "microvm_boot_id": MICROVM_BOOT_ID,
        "tools_used_in_session": tools_used,
    }


if __name__ == "__main__":
    app.run()

## 4. Runtime 에 JWT 인증 적용 후 재배포 (슬라이드 21)

- `authorizer_configuration.customJWTAuthorizer` : Cognito **discovery URL** 과 허용할 **client ID**
- `request_header_configuration` : `Authorization` 헤더를 에이전트 코드까지 전달하도록 허용 (기본은 차단)

같은 이름으로 다시 배포하면 런타임이 **업데이트되고 새 버전(V2)** 이 생깁니다. `DEFAULT` 는 V2 로 이동하지만 `PROD` 는 V1 에 그대로 고정됩니다.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

state = config.load_state()
agentcore_runtime = Runtime()
agentcore_runtime.configure(
    entrypoint="agent/runtime_ch03.py",
    agent_name=config.AGENT_NAME,
    execution_role=state["runtime_role_arn"],
    auto_create_ecr=True,
    requirements_file="agent/requirements.txt",
    region=REGION,
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": state["cognito_discovery_url"],
            "allowedClients": [state["cognito_client_id"]],
        }
    },
    request_header_configuration={"requestHeaderAllowlist": ["Authorization"]},
)
launch_result = agentcore_runtime.launch(auto_update_on_conflict=True)
wait_for(lambda: agentcore_runtime.status().endpoint["status"],
         ready={"READY"}, failed={"CREATE_FAILED", "UPDATE_FAILED"}, interval=10, label="Runtime DEFAULT 엔드포인트")
print("최신 버전:", control.get_agent_runtime(agentRuntimeId=agent_id)["agentRuntimeVersion"])

## 5. 인증 테스트

### 5-1. 거부되어야 하는 호출들

In [ ]:
# (1) IAM SigV4 — 이제 JWT 전용 런타임이므로 거부
try:
    invoke_with_iam(agent_arn, "안녕하세요", new_session_id())
except Exception as e:
    print("❌ SigV4 호출 거부:", str(e)[:200])

# (2) 토큰 없음
print("❌ 토큰 없음  :", invoke_with_jwt(agent_arn, "안녕하세요", None, new_session_id(), raise_on_error=False))

# (3) 위조/변조된 토큰
forged = alice_token[:-10] + "AAAAAAAAAA"
print("❌ 변조 토큰  :", invoke_with_jwt(agent_arn, "안녕하세요", forged, new_session_id(), raise_on_error=False))

### 5-2. 로그인한 고객 alice 의 호출 — 사용자 컨텍스트 + 아웃바운드 API 키

In [ ]:
alice_session = new_session_id("alice")
show(invoke_with_jwt(agent_arn, "안녕하세요! 제가 누구로 로그인되어 있나요? 그리고 주문 ORD-1001 배송 상태 알려주세요.",
                     alice_token, alice_session))

👉 응답의 `파트너 API 키 ****xxxx` 끝 4자리가 2단계에서 출력한 값과 같은지 확인하세요.
에이전트 코드에는 키가 없지만, **워크로드 자격 증명 → AgentCore Identity → 토큰 볼트** 경로로 실행 시점에 안전하게 전달되었습니다.

### 5-3. 세션 소유자 검증 — bob 이 alice 의 세션 ID 를 재사용하면?

In [ ]:
show(invoke_with_jwt(agent_arn, "이전 대화 내용 보여줘", bob_token, alice_session))

In [ ]:
# bob 은 자신의 세션으로 정상 이용
show(invoke_with_jwt(agent_arn, "제가 누구죠? 주문 ORD-2002 배송 조회해 주세요.", bob_token, new_session_id("bob")))

## 6. 워크로드 자격 증명(Workload Identity) 확인 (슬라이드 16–17)

Runtime 에 배포하면 에이전트 전용 **워크로드 자격 증명**이 자동 생성됩니다.
에이전트는 이 ID 로 *"이 에이전트가 사용자 X 를 대신해 리소스 Y 에 접근"* 하는 **워크로드 액세스 토큰**을 받습니다.

In [ ]:
for wi in control.list_workload_identities().get("workloadIdentities", []):
    if config.AGENT_NAME in (wi.get("name") or wi.get("workloadIdentityName") or ""):
        print(json.dumps(wi, indent=2, default=str))

## 7. 런타임 실행 역할 살펴보기 (슬라이드 22)

02장에서 만든 실행 역할의 권한입니다. 각 Statement 가 어떤 AgentCore 기능과 연결되는지 확인해 보세요.

In [ ]:
iam = boto3.client("iam")
doc = iam.get_role_policy(RoleName=state["runtime_role_name"], PolicyName="AgentCoreDemoRuntimePolicy")["PolicyDocument"]
for stmt in doc["Statement"]:
    actions = stmt["Action"] if isinstance(stmt["Action"], list) else [stmt["Action"]]
    print(f"[{stmt.get('Sid', '-')}]")
    for a in actions:
        print("   ", a)

## 8. 감사 추적 확인 (슬라이드 10)

에이전트가 출력한 `AUDIT` 로그는 런타임 로그 그룹(`/aws/bedrock-agentcore/runtimes/<agent_id>-DEFAULT`)에 저장됩니다.
*누가, 어느 세션에서, 무엇을 요청했고, 어떤 도구를 썼는지* 추적할 수 있습니다. (로그 반영까지 1~2분 걸릴 수 있습니다.)

In [ ]:
logs = boto3.client("logs", region_name=REGION)
log_group = f"/aws/bedrock-agentcore/runtimes/{agent_id}-DEFAULT"

events = logs.filter_log_events(
    logGroupName=log_group, filterPattern='"AUDIT"',
    startTime=int((time.time() - 1800) * 1000),
)["events"]
for e in events[-10:]:
    msg = e["message"]
    print(msg[msg.find("{"):] if "{" in msg else msg)

## 9. 참고: `PROD` 엔드포인트는 아직 V1
버전은 배포 시점의 **코드 + 구성(인증 방식 포함)의 스냅샷**입니다. `PROD` 는 여전히 V1(IAM 인증, 인증 기능 없는 코드)을 서비스합니다.
06장에서 품질 평가를 마친 최종 버전으로 승격합니다.

In [ ]:
for ep in control.list_agent_runtime_endpoints(agentRuntimeId=agent_id).get("runtimeEndpoints", []):
    print(f"{ep.get('name'):8} → liveVersion={ep.get('liveVersion')}")

## ✅ Chapter 03 정리

| 달성한 것 | 아직 부족한 것 (→ 다음 챕터) |
|---|---|
| Cognito JWT 로 **인증된 고객만** 호출 (SigV4·무토큰·변조 토큰 거부) | 보증 조회·환불 등 **사내 시스템 도구**가 없음 |
| JWT 클레임 기반 사용자 컨텍스트, 세션 소유자 검증, 감사 로그 | 도구가 에이전트 코드에 묶여 **다른 에이전트와 공유 불가** |
| AgentCore Identity 로 **외부 API 키를 코드 밖에서** 안전하게 사용 | 환불 금액 같은 **비즈니스 규칙을 강제할 장치 없음** |

➡️ **[04_ToolsGateway_Demo](04_ToolsGateway_Demo.ipynb)**: 사내 Lambda 를 AgentCore Gateway 로 MCP 도구화하고,
지금 만든 Cognito 토큰을 **그대로 Gateway 까지 전파**해 도구 접근을 인증하며, AgentCore Policy 로 환불 한도를 강제합니다.